## Data Cleaning



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/raw/pollution_sample.csv", parse_dates=['Date Local'])

### Data Preparation

In [3]:
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

In [4]:
for c in ['State Code', 'County Code', 'Site Num', 'Year']:
    df[c] = df[c].astype(int)

In [5]:
df = df.sort_values(['station', 'Date Local']).reset_index(drop=True)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70922 entries, 0 to 70921
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   station            70922 non-null  object        
 1   State              70922 non-null  object        
 2   City               70922 non-null  object        
 3   Date Local         70922 non-null  datetime64[ns]
 4   State Code         70922 non-null  int64         
 5   County Code        70922 non-null  int64         
 6   Site Num           70922 non-null  int64         
 7   NO2 Mean           70922 non-null  float64       
 8   NO2 1st Max Value  70922 non-null  float64       
 9   NO2 1st Max Hour   70922 non-null  float64       
 10  NO2 AQI            70922 non-null  float64       
 11  O3 Mean            70922 non-null  float64       
 12  O3 1st Max Value   70922 non-null  float64       
 13  O3 1st Max Hour    70922 non-null  float64       
 14  O3 AQI

In [7]:
print(df.duplicated(['station', 'Date Local']).sum())

0


In [8]:
miss = df.isna().sum()
print(miss[miss > 0])
print((miss / len(df) * 100).round(3)[miss > 0])

CO AQI    43
dtype: int64
CO AQI    0.061
dtype: float64


In [9]:
# Do NOT impute pollutant values in the general cleaning stage.
# Missing AQI values are preserved here and handled later in the ML section
# using past-only forward filling, after the station calendars are completed.
print("Missing pollutant values are preserved for leakage-safe ML preprocessing:")
print(df[['NO2 AQI', 'O3 AQI', 'SO2 AQI', 'CO AQI']].isna().sum())


Missing pollutant values are preserved for leakage-safe ML preprocessing:
NO2 AQI     0
O3 AQI      0
SO2 AQI     0
CO AQI     43
dtype: int64


In [10]:
neg_cols = ['NO2 Mean','NO2 1st Max Value','SO2 Mean','SO2 1st Max Value',
            'CO Mean','CO 1st Max Value']
print((df[neg_cols] < 0).sum())

NO2 Mean              36
NO2 1st Max Value     12
SO2 Mean             645
SO2 1st Max Value     68
CO Mean              115
CO 1st Max Value      11
dtype: int64


In [11]:
df[neg_cols] = df[neg_cols].clip(lower=0)

In [12]:
cols = [
    'NO2 1st Max Hour', 'O3 1st Max Hour',
    'SO2 1st Max Hour', 'CO 1st Max Hour',
    'NO2 AQI', 'O3 AQI', 'SO2 AQI', 'CO AQI'
]

df[cols] = df[cols].round().astype('Int64')


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70922 entries, 0 to 70921
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   station            70922 non-null  object        
 1   State              70922 non-null  object        
 2   City               70922 non-null  object        
 3   Date Local         70922 non-null  datetime64[ns]
 4   State Code         70922 non-null  int64         
 5   County Code        70922 non-null  int64         
 6   Site Num           70922 non-null  int64         
 7   NO2 Mean           70922 non-null  float64       
 8   NO2 1st Max Value  70922 non-null  float64       
 9   NO2 1st Max Hour   70922 non-null  Int64         
 10  NO2 AQI            70922 non-null  Int64         
 11  O3 Mean            70922 non-null  float64       
 12  O3 1st Max Value   70922 non-null  float64       
 13  O3 1st Max Hour    70922 non-null  Int64         
 14  O3 AQI

In [14]:
for c in ['State', 'City']:
    df[c] = df[c].str.strip()

check = df.groupby('station')[['State', 'City']].nunique()
print(check[(check['State'] > 1) | (check['City'] > 1)])   

rebuilt = (df['State Code'].astype(str) + '_' +
           df['County Code'].astype(str) + '_' +
           df['Site Num'].astype(str))
print((rebuilt == df['station']).all())  

print(df['State'].nunique(), df['City'].nunique(), df['station'].nunique())

Empty DataFrame
Columns: [State, City]
Index: []
True
29 32 37


In [15]:
df['Month'] = df['Date Local'].dt.month
df['Season'] = (df['Month'] % 12 // 3).map({0: 'Winter', 1: 'Spring', 2: 'Summer', 3: 'Fall'})

In [16]:
print(df[['Date Local', 'Month', 'Season']].head())

  Date Local  Month  Season
0 2010-01-01      1  Winter
1 2010-01-02      1  Winter
2 2010-01-03      1  Winter
3 2010-01-04      1  Winter
4 2010-01-05      1  Winter


In [17]:
region_map = {
    'Northeast': ['Maine', 'New Hampshire', 'Vermont', 'Massachusetts', 'Rhode Island',
                  'Connecticut', 'New York', 'New Jersey', 'Pennsylvania'],
    'Midwest':   ['Ohio', 'Michigan', 'Indiana', 'Illinois', 'Wisconsin', 'Minnesota',
                  'Iowa', 'Missouri', 'North Dakota', 'South Dakota', 'Nebraska', 'Kansas'],
    'South':     ['Delaware', 'Maryland', 'Virginia', 'West Virginia', 'North Carolina',
                  'South Carolina', 'Georgia', 'Florida', 'Kentucky', 'Tennessee', 'Alabama',
                  'Mississippi', 'Arkansas', 'Louisiana', 'Oklahoma', 'Texas',
                  'District Of Columbia'],
    'West':      ['Montana', 'Idaho', 'Wyoming', 'Colorado', 'New Mexico', 'Arizona', 'Utah',
                  'Nevada', 'California', 'Oregon', 'Washington', 'Alaska', 'Hawaii']}


In [18]:
state2region = {s: r for r, states in region_map.items() for s in states}
df['Region'] = df['State'].map(state2region)

In [19]:
print(df['Region'].isna().sum())
print(df['Region'].value_counts())

0
Region
South        21332
Northeast    19480
West         18685
Midwest      11425
Name: count, dtype: int64


### Units
- NO2, SO2: ppb
- O3, CO: ppm
- AQI: unitless (0-500)
- 1st Max Hour: hour of day (0-23)

In [20]:
df_clean.to_csv("../data/processed/df_clean.csv", index=False)

NameError: name 'df_clean' is not defined